# AI作曲アレンジ — 音を聴き、設定を変え、伴奏を作ろう

**YouTube・手持ちの音声・デモ音源から、コード伴奏とドラムを加えて聴き比べる教材です。**

2026年度3Q（月3木3クラス）の学生2名による作品をもとに、公開用に整理・改良しました。
音源を解析し、ボタンや選択欄で音作りを試せる、元の作品の工夫を引き継いでいます。

## 最初の操作

1. Google Colabで開き、編集・保存する場合は「ファイル → ドライブにコピーを保存」を選びます。
2. 「ランタイム → すべてのセルを実行」で、下の **1 → 2 → 3** を実行します。CPUのままで使えます。
3. 最後の操作画面で **「デモ音源を用意」→「音源を解析」→「アレンジを生成」** と進みます。
4. 原曲・追加パート・ミックスを聴き比べ、WAVとMIDIをダウンロードします。

初回の準備や解析には数分かかることがあります。左の実行マークが止まるまで待ってください。
Notebook内に全コードを収録しているので、このファイルだけでも実行できます。

[詳しい操作マニュアル](https://github.com/shogo-ishikawa/2026_3Q_M3Th3/blob/main/projects/ai-music-arranger/docs/manual.md)
／ [コードと改良内容](https://github.com/shogo-ishikawa/2026_3Q_M3Th3/tree/main/projects/ai-music-arranger)

### できることと、結果の読み方

- YouTube URLから、開始位置と長さ（3〜60秒）を指定して音声を取得できます。
- BPMとキーを推定し、pYINで連続した音の高さを音符に変換します。推定は耳で確かめ、修正できます。
- キー・スケール・楽器・各パートの音量を変えて、規則に基づく伴奏を作ります。
- 「AI作曲アレンジ」という作品名を引き継いでいますが、大規模な音楽生成AIやLSTMは使いません。
  ボーカルと伴奏を分離したり、原曲のコード進行を正確に認識したりする機能ではありません。
- BPMは追加パートの速さを変えます。原曲の再生速度は変わりません。
- 試聴は内蔵の簡易音源です。MIDIの音色は再生ソフトによって異なります。

音源は、自分で制作したものや利用が認められたものを使ってください。
YouTube側の接続制限で取得できない場合もあります。失敗時には理由を表示し、別の音源を選べます。

## 1. 環境を準備する

左の ▶ を押します。音声解析ライブラリ、FFmpeg、YouTube取得に使う実行環境を確認します。
このセルは通常、接続したランタイムごとに1回だけ実行します。APIキーやGPUは不要です。
YouTube用の追加準備に失敗した場合は、その案内を確認してください。デモとファイル入力は試せます。

In [ ]:
"""Colab の初回準備。ノートブックにも同じコードを収録します。"""

from pathlib import Path
import hashlib
import importlib
import os
import platform
import re
import shutil
import subprocess
import sys
import tempfile
import urllib.request
import zipfile


PACKAGES = [
    "librosa>=0.11,<0.12", "soundfile>=0.13,<0.14",
    "pretty_midi>=0.2.10,<0.3", "ipywidgets>=8.1,<9",
    "matplotlib>=3.9,<4",
]
DENO_VERSION = "v2.9.5"


def supported_js_runtime():
    """yt-dlp が対応する JavaScript 実行環境があるか調べます。"""
    for name, minimum in [("deno", (2, 3, 0)), ("node", (22, 0, 0))]:
        executable = shutil.which(name)
        if not executable:
            continue
        try:
            result = subprocess.run(
                [executable, "--version"], capture_output=True,
                text=True, timeout=10, check=True,
            )
            match = re.search(r"(\d+)\.(\d+)\.(\d+)", result.stdout)
            if match and tuple(map(int, match.groups())) >= minimum:
                return name
        except (OSError, subprocess.SubprocessError):
            continue
    return None


def install_deno(bin_dir):
    """公式配布の Deno をチェックサムで照合して配置します（Linux Colab 用）。"""
    if platform.system() != "Linux" or platform.machine() not in {"x86_64", "AMD64"}:
        raise RuntimeError("この環境では Deno または Node.js を手動で導入してください。")
    filename = "deno-x86_64-unknown-linux-gnu.zip"
    base = f"https://github.com/denoland/deno/releases/download/{DENO_VERSION}/"
    with urllib.request.urlopen(base + filename + ".sha256sum", timeout=30) as response:
        expected = response.read(4096).decode("utf-8").split()[0]
    if not re.fullmatch(r"[a-fA-F0-9]{64}", expected):
        raise RuntimeError("Deno のチェックサムを確認できませんでした。")
    bin_dir = Path(bin_dir)
    bin_dir.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=bin_dir) as temporary:
        archive = Path(temporary) / filename
        size = 0
        with urllib.request.urlopen(base + filename, timeout=60) as response, archive.open("wb") as out:
            while chunk := response.read(1024 * 1024):
                size += len(chunk)
                if size > 100 * 1024 * 1024:
                    raise RuntimeError("Deno の取得サイズが想定を超えました。")
                out.write(chunk)
        if hashlib.sha256(archive.read_bytes()).hexdigest() != expected.lower():
            raise RuntimeError("Deno の照合に失敗しました。時間をおいて準備セルを再実行してください。")
        with zipfile.ZipFile(archive) as zipped:
            executable = Path(temporary) / "deno"
            executable.write_bytes(zipped.read("deno"))
        executable.chmod(0o755)
        executable.replace(bin_dir / "deno")
    os.environ["PATH"] = str(bin_dir.resolve()) + os.pathsep + os.environ.get("PATH", "")


def setup_environment():
    """準備が済んだら、セル 2 のプログラムを保存する場所へ移動します。"""
    try:
        import google.colab  # noqa: F401
        in_colab = True
    except ImportError:
        in_colab = False
    base = Path("/content") if in_colab else Path.cwd()
    runtime_dir = base if base.name == ".ai_music_arranger_runtime" else base / ".ai_music_arranger_runtime"
    runtime_dir.mkdir(parents=True, exist_ok=True)
    print("1/3 音声解析と操作画面に必要なライブラリを準備しています。")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES],
        check=True, timeout=360,
    )
    # YouTubeの仕様変更に備え、取得ツールと対応するEJSを一緒に更新します。
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "yt-dlp[default]"],
        check=True, timeout=180,
    )
    importlib.invalidate_caches()

    print("2/3 音声を変換する FFmpeg を確認しています。")
    if not shutil.which("ffmpeg"):
        if not in_colab:
            raise RuntimeError("FFmpeg がありません。README のローカル実行手順を確認してください。")
        subprocess.run(["apt-get", "-qq", "update"], check=True, timeout=120)
        subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True, timeout=180)

    bin_dir = runtime_dir / "bin"
    if bin_dir.exists():
        os.environ["PATH"] = str(bin_dir.resolve()) + os.pathsep + os.environ.get("PATH", "")
    print("3/3 YouTube 取得に必要な JavaScript 実行環境を確認しています。")
    if not supported_js_runtime():
        try:
            install_deno(bin_dir)
        except Exception:
            print("YouTube 用の追加準備が完了しませんでした。準備セルを再実行してください。")
            print("デモ音源と PC からのファイル読み込みはそのまま試せます。")
    runtime = supported_js_runtime()
    print(f"YouTube 用実行環境: {runtime or '未準備（操作マニュアルの対処方法を参照）'}")

    if in_colab:
        from google.colab import output
        output.enable_custom_widget_manager()
    os.chdir(runtime_dir)
    if str(runtime_dir.resolve()) not in sys.path:
        sys.path.insert(0, str(runtime_dir.resolve()))
    print("準備が完了しました。続けてセル 2 の各プログラムと、セル 3 を実行してください。")


if __name__ == "__main__":
    setup_environment()


## 2. プログラムを読み込む（3つのセル）

続く3つのセルは、役割ごとに分けたPythonコードを作業用フォルダーへ保存します。
**3つとも上から順に実行**してください。実行すると `Writing ...py` と表示されます。

初めて使うときは、コードを変更せずに進めて構いません。
処理を学ぶ場合は、まず音の間隔を並べた `SCALE_DEFS`、チェックボックスに対応する `if`、
音符を並べる `for`、処理をまとめた `def` を探してみてください。

### 2-a. 音源の取り込み

YouTubeのURLを確認し、指定した区間だけを取り込みます。失敗したときは、前の音源を保持します。

In [ ]:
%%writefile audio_input.py
"""音源の取り込み。YouTube取得と音声の切り出しを音楽処理から分けます。"""

from pathlib import Path
import contextlib
import math
import os
import re
import shutil
import signal
import subprocess
import sys
import tempfile
from urllib.parse import parse_qs, urlsplit

import numpy as np
import soundfile as sf


SAMPLE_RATE = 22050
MIN_DURATION = 3.0
MAX_DURATION = 60.0
DOWNLOAD_TIMEOUT = 180
DECODE_TIMEOUT = 60


class AudioInputError(ValueError):
    """利用者が次の操作を判断できる、短い取り込みエラー。"""


def _run_process(command, timeout, text=False):
    """Colabでは子のFFmpegも同じグループに置き、時間切れ時に止めます。"""
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                               text=text, start_new_session=(os.name == "posix"))
    try:
        stdout, stderr = process.communicate(timeout=timeout)
    except BaseException:
        # yt-dlpだけを止めると、その子のFFmpegが通信を続ける場合があります。
        with contextlib.suppress(ProcessLookupError):
            if os.name == "posix":
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
        process.communicate()
        raise
    return subprocess.CompletedProcess(command, process.returncode, stdout, stderr)


def normalise_youtube_url(url):
    """通常の動画・短縮URL・Shorts・ライブのURLを単一動画URLに直します。"""
    if not isinstance(url, str):
        raise AudioInputError("YouTubeの動画URLを入力してください。")
    value = url.strip()
    try:
        parts = urlsplit(value)
        # 不正なポート番号もここで検出します。
        port = parts.port
    except ValueError:
        raise AudioInputError("YouTubeの動画URLを確認してください。") from None
    if (parts.scheme != "https" or parts.username or parts.password
            or port not in (None, 443)):
        raise AudioInputError("https:// で始まるYouTubeの動画URLを入力してください。")
    host = (parts.hostname or "").lower()
    if host == "youtu.be":
        video_id = parts.path.strip("/")
    elif host in {"youtube.com", "www.youtube.com", "m.youtube.com", "music.youtube.com"}:
        if parts.path == "/watch":
            ids = parse_qs(parts.query).get("v", [])
            video_id = ids[0] if len(ids) == 1 else ""
        else:
            match = re.fullmatch(r"/(?:shorts|live|embed)/([A-Za-z0-9_-]{11})/?", parts.path)
            video_id = match.group(1) if match else ""
    else:
        raise AudioInputError("YouTubeまたはyoutu.beの動画URLを入力してください。")
    if not re.fullmatch(r"[A-Za-z0-9_-]{11}", video_id):
        raise AudioInputError("単一動画のURLを入力してください。チャンネルや再生リストは使えません。")
    return "https://www.youtube.com/watch?v=" + video_id


def _validate_section(start, duration):
    try:
        start, duration = float(start), float(duration)
    except (ValueError, TypeError):
        raise AudioInputError("開始位置と長さは秒数で入力してください。") from None
    if not math.isfinite(start) or start < 0:
        raise AudioInputError("開始位置は0秒以上にしてください。")
    if not math.isfinite(duration) or not MIN_DURATION <= duration <= MAX_DURATION:
        raise AudioInputError("取り込む長さは3〜60秒にしてください。")
    return start, duration


def find_js_runtime():
    """yt-dlpが対応するDenoまたはNodeを探します。見つからなければNone。"""
    for name, minimum in (("deno", (2, 3)), ("node", (22, 0))):
        path = shutil.which(name)
        if not path:
            continue
        try:
            result = subprocess.run([path, "--version"], capture_output=True,
                                    text=True, timeout=10, check=False)
            match = re.search(r"(?:^|\s|v)(\d+)\.(\d+)", result.stdout)
            if result.returncode == 0 and match:
                version = tuple(int(x) for x in match.groups())
                if version >= minimum:
                    return name, path
        except (OSError, subprocess.TimeoutExpired):
            continue
    return None


def load_audio_file(path, start=0.0, duration=20.0, sr=SAMPLE_RATE):
    """FFmpegで指定区間だけを読み、モノラルのfloat32配列を返します。

    取り込む長さは3〜60秒。曲末まではそれより短くても読み込めます。
    ファイル全体をメモリに展開せず、変換は最大60秒で打ち切ります。
    """
    start, duration = _validate_section(start, duration)
    if isinstance(sr, bool) or not isinstance(sr, int) or not 8000 <= sr <= 48000:
        raise AudioInputError("サンプリング周波数は8000〜48000 Hzの整数にしてください。")
    source = Path(path)
    if not source.is_file():
        raise AudioInputError("音声ファイルが見つかりません。ファイルを選び直してください。")
    ffmpeg = shutil.which("ffmpeg")
    if not ffmpeg:
        raise AudioInputError("FFmpegがありません。先に環境セットアップのセルを実行してください。")
    command = [ffmpeg, "-nostdin", "-hide_banner", "-loglevel", "error",
               "-ss", f"{start:.6f}", "-i", str(source.resolve()),
               "-t", f"{duration:.6f}", "-vn", "-ac", "1", "-ar", str(sr),
               "-f", "f32le", "pipe:1"]
    try:
        result = _run_process(command, timeout=DECODE_TIMEOUT)
    except subprocess.TimeoutExpired:
        raise AudioInputError("音声の変換が60秒以内に終わりませんでした。短い区間で試してください。") from None
    except OSError:
        raise AudioInputError("音声の変換を開始できません。環境セットアップをやり直してください。") from None
    if result.returncode != 0 or len(result.stdout) % 4:
        raise AudioInputError("音声を読み込めません。WAV・MP3などの音声ファイルを選んでください。")
    samples = np.frombuffer(result.stdout, dtype="<f4").copy()
    # 万一デコーダーが長めに返しても、指定区間のサンプル数に制限します。
    samples = samples[:int(round(duration * sr))]
    if samples.size == 0 or not np.all(np.isfinite(samples)):
        raise AudioInputError("指定した区間に読み込める音声がありません。開始位置を確認してください。")
    return samples


def download_youtube(url, destination, start=0.0, duration=20.0):
    """指定区間をYouTubeから取得し、22050 Hz・モノラルWAVとして保存。

    yt-dlpには指定した区間だけを要求します。配信方式によって通信量が
    区間長に比例するとは限りません。取得は最大180秒、続く変換は最大60秒。
    失敗時には、前回成功したdestinationを変更しません。
    """
    canonical_url = normalise_youtube_url(url)
    start, duration = _validate_section(start, duration)
    if not shutil.which("ffmpeg"):
        raise AudioInputError("FFmpegがありません。先に環境セットアップのセルを実行してください。")
    runtime = find_js_runtime()
    if runtime is None:
        raise AudioInputError("YouTube取得にはDeno 2.3以上またはNode 22以上が必要です。環境セットアップをやり直してください。")
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    name, executable = runtime
    with tempfile.TemporaryDirectory(prefix="youtube-", dir=destination.parent) as staging:
        staging = Path(staging)
        template = staging / "source.%(ext)s"
        command = [sys.executable, "-m", "yt_dlp", "--ignore-config", "--no-playlist",
                   "--no-progress", "--no-warnings", "--socket-timeout", "20",
                   "--retries", "2", "--fragment-retries", "2",
                   "--js-runtimes", f"{name}:{executable}", "-f", "bestaudio/best",
                   "--download-sections", f"*{start:.6f}-{start + duration:.6f}",
                   "-x", "--audio-format", "wav", "-o", str(template)]
        if os.environ.get("SSL_CERT_FILE"):
            # 明示的に設定されたCAを尊重します。TLSの証明書・ホスト名検証は継続。
            command += ["--compat-options", "no-certifi"]
        command += ["--", canonical_url]
        try:
            result = _run_process(command, text=True, timeout=DOWNLOAD_TIMEOUT)
        except subprocess.TimeoutExpired:
            raise AudioInputError("YouTubeからの取得が180秒以内に終わりませんでした。短い区間で再試行してください。") from None
        except OSError:
            raise AudioInputError("YouTube取得を開始できません。環境セットアップをやり直してください。") from None
        source = staging / "source.wav"
        if result.returncode != 0 or not source.is_file():
            raise AudioInputError("YouTubeから取得できませんでした。動画の公開状態とURLを確認し、セットアップを再実行してから試してください。YouTube側で接続を拒否される場合は、お手持ちの音源でも操作できます。")
        samples = load_audio_file(source, start=0.0, duration=duration)
        converted = staging / "validated.wav"
        try:
            sf.write(converted, samples, SAMPLE_RATE, subtype="PCM_16")
            os.replace(converted, destination)
        except (OSError, RuntimeError):
            raise AudioInputError("取得した音源を保存できません。保存先の空き容量を確認してください。") from None
    return destination


### 2-b. 解析と伴奏生成

テンポ・キー・音の高さを推定し、コードとドラムを並べます。
伴奏の規則を変更したいときは、このセルにある `chord_pitches` や `progression` を読んでみましょう。

In [ ]:
%%writefile music_core.py
"""音源から伴奏を作る、画面表示と分離した処理。

2026年度3Qの学生作品にあったpYIN・コード伴奏・ドラムを引き継ぐ。
音声の分離モデルや大規模な生成モデルは使わず、推定と規則で編曲する。
"""
from dataclasses import dataclass
import math

import librosa
import numpy as np
import pretty_midi

SAMPLE_RATE = 22050
NOTE_NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
SCALE_DEFS = {
    "major": [0, 2, 4, 5, 7, 9, 11],
    "natural_minor": [0, 2, 3, 5, 7, 8, 10],
    "harmonic_minor": [0, 2, 3, 5, 7, 8, 11],
    "dorian": [0, 2, 3, 5, 7, 9, 10],
    "mixolydian": [0, 2, 4, 5, 7, 9, 10],
    "pentatonic": [0, 2, 4, 7, 9],
}
INSTRUMENTS = {"Electric Piano": 4, "Acoustic Grand Piano": 0,
               "Acoustic Guitar": 24, "Electric Guitar": 27,
               "Strings": 48, "Synth Lead": 80, "Music Box": 10}


@dataclass(frozen=True)
class Note:
    pitch: int
    start: float
    end: float
    velocity: int = 90


@dataclass
class ArrangementConfig:
    bpm: float = 120.0
    root: str = "C"
    scale: str = "major"
    instrument: str = "Electric Piano"
    add_chords: bool = True
    add_drums: bool = True
    add_melody: bool = True
    original_gain: float = 0.7
    chord_gain: float = 0.6
    drum_gain: float = 0.8
    melody_gain: float = 0.6
    beat_offset: float = 0.0
    humanise: float = 0.0
    seed: int = 42


def _audio(y, sr):
    y = np.asarray(y, dtype=np.float64)
    if y.ndim != 1 or y.size == 0 or not np.isfinite(y).all():
        raise ValueError("音源は空でない、有限の数値からなるモノラル音声にしてください。")
    if isinstance(sr, bool) or not isinstance(sr, (int, np.integer)) or not 8000 <= sr <= 192000:
        raise ValueError("サンプリング周波数は8000〜192000 Hzの整数にしてください。")
    return y


def _validate(config):
    if config.root not in NOTE_NAMES or config.scale not in SCALE_DEFS:
        raise ValueError("キーまたはスケールが選択肢にありません。")
    if config.instrument not in INSTRUMENTS:
        raise ValueError("楽器が選択肢にありません。")
    for name, low, high in [("bpm", 40, 240), ("beat_offset", 0, 60),
                            ("humanise", 0, 1), ("original_gain", 0, 2),
                            ("chord_gain", 0, 2), ("drum_gain", 0, 2),
                            ("melody_gain", 0, 2)]:
        value = getattr(config, name)
        if not isinstance(value, (int, float, np.number)) or not math.isfinite(value) or not low <= value <= high:
            raise ValueError(f"{name}は{low}〜{high}の有限の数値にしてください。")
    if not isinstance(config.seed, (int, np.integer)) or not 0 <= config.seed < 2**32:
        raise ValueError("乱数シードは0〜4294967295の整数にしてください。")


def make_demo(duration=12.0, sr=SAMPLE_RATE):
    """外部の曲を使わない、C長調・120 BPMの短い動作確認用フレーズ。"""
    if not isinstance(duration, (int, float)) or not math.isfinite(duration) or not 0.2 <= duration <= 60:
        raise ValueError("デモの長さは0.2〜60秒にしてください。")
    y = np.zeros(round(duration * sr))
    _audio(y, sr)
    pitches = [60, 64, 67, 72, 65, 64, 62, 60, 60, 64, 65, 69, 67, 65, 62, 60]
    for i, start in enumerate(np.arange(0, duration, 0.5)):
        left, right = round(start * sr), min(len(y), round((start + 0.45) * sr))
        t = np.arange(right - left) / sr
        freq = librosa.midi_to_hz(pitches[i % len(pitches)])
        envelope = np.minimum(t / 0.012, 1) * np.minimum((len(t) / sr - t) / 0.035, 1)
        y[left:right] += 0.4 * envelope * np.exp(-1.5 * t) * np.sin(2 * np.pi * freq * t)
    return y.astype(np.float32)


def analyse_audio(y, sr=SAMPLE_RATE):
    """BPM・拍の位相・長調/短調を推定する。confidenceは比較用の目安。"""
    y = _audio(y, sr)
    result = dict(bpm=120.0, key="C", scale="major", beat_offset=0.0,
                  confidence=0.0, note="推定値は目安です。原曲を聴きながら修正してください。")
    if len(y) < sr * 0.12 or np.max(np.abs(y)) < 1e-6:
        result["note"] = "音源が短いか無音のため、仮の設定（C長調・120 BPM）を表示しています。"
        return result
    tempo, beats = librosa.beat.beat_track(y=y, sr=sr, trim=False)
    bpm = float(np.asarray(tempo).reshape(-1)[0])
    if math.isfinite(bpm) and bpm > 0:
        while bpm < 60:
            bpm *= 2
        while bpm > 200:
            bpm /= 2
        result["bpm"] = round(bpm, 2)
        if len(beats):
            first = float(librosa.frames_to_time(beats[0], sr=sr))
            result["beat_offset"] = round(first % (60 / bpm), 3)
    # 一番大きな音を主音とせず、12音全体と調のプロファイルを比較する。
    padded = np.pad(y, (0, max(0, 4096 - len(y))))
    chroma = librosa.feature.chroma_stft(y=padded, sr=sr, n_fft=4096).mean(axis=1)
    profiles = {
        "major": np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88]),
        "natural_minor": np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17]),
    }
    scores = []
    if np.std(chroma) > 1e-8:
        for mode, profile in profiles.items():
            for root in range(12):
                score = float(np.corrcoef(chroma, np.roll(profile, root))[0, 1])
                scores.append((score, root, mode))
        scores.sort(reverse=True)
        best, root, mode = scores[0]
        result.update(key=NOTE_NAMES[root], scale=mode,
                      confidence=round(max(0.0, best - scores[1][0]), 3))
    return result


def pitches_to_notes(f0, voiced, sr, hop_length, duration, min_duration=0.08):
    """ピッチの連続区間を音符にする。最後に続いている音も確定する。"""
    notes = []
    current, start = None, 0.0
    for index, (freq, is_voiced) in enumerate(zip(f0, voiced)):
        t = min(index * hop_length / sr, duration)
        pitch = int(round(librosa.hz_to_midi(freq))) if is_voiced and np.isfinite(freq) and freq > 0 else None
        if pitch is not None and not 0 <= pitch <= 127:
            pitch = None
        if pitch != current:
            if current is not None and t - start >= min_duration:
                notes.append(Note(current, start, t))
            current, start = pitch, t
    if current is not None and duration - start >= min_duration:
        notes.append(Note(current, start, duration))
    return notes


def extract_melody(y, sr=SAMPLE_RATE):
    """pYINで単音の高さを推定。多重奏の主旋律を分離する処理ではない。"""
    y = _audio(y, sr)
    if len(y) < sr * 0.08 or np.max(np.abs(y)) < 1e-6:
        return []
    frame_length, hop_length = 2048, 512
    padded = np.pad(y, (0, max(0, frame_length - len(y))))
    f0, voiced, _ = librosa.pyin(padded, fmin=librosa.note_to_hz("C3"),
                                fmax=min(librosa.note_to_hz("C6"), sr / 2 - 1),
                                sr=sr, frame_length=frame_length, hop_length=hop_length,
                                n_thresholds=50, resolution=0.2)
    return pitches_to_notes(f0, voiced, sr, hop_length, len(y) / sr)


def chord_pitches(root, scale, degree):
    """スケール上の音を1音おきに3つ重ねる。degreeは0から数える。"""
    intervals = SCALE_DEFS[scale]
    tonic = 48 + NOTE_NAMES.index(root)
    pitches = []
    for step in (0, 2, 4):
        octave, index = divmod(degree + step, len(intervals))
        pitches.append(tonic + 12 * octave + intervals[index])
    return pitches


def _scaled_note(pitch, start, end, velocity, gain, duration, rng, humanise):
    if gain <= 0 or velocity <= 0:
        return None
    start = max(0, start + rng.uniform(-0.018, 0.018) * humanise)
    end = min(duration, end)
    velocity = int(np.clip(round((velocity + rng.uniform(-10, 10) * humanise) * gain), 1, 127))
    if start >= end:
        return None
    return Note(int(pitch), float(start), float(end), velocity)


def _render(notes, sr, sample_count, program=4, drums=False, seed=42):
    """サンプル音源なしで聴ける簡易合成。GM音源の忠実な再現ではない。"""
    audio = np.zeros(sample_count)
    rng = np.random.default_rng(seed)
    for note in notes:
        left, right = round(note.start * sr), min(sample_count, round(note.end * sr))
        if right <= left:
            continue
        t = np.arange(right - left) / sr
        if drums:
            if note.pitch == 36:
                wave = np.sin(2 * np.pi * (48 * t + 24 * (1 - np.exp(-30 * t)) / 30)) * np.exp(-22 * t)
            else:
                noise = rng.uniform(-1, 1, len(t))
                wave = noise * np.exp(-(65 if note.pitch == 42 else 28) * t)
            amplitude = 0.28
        else:
            frequency = librosa.midi_to_hz(note.pitch)
            # 倍音を変えることで音色の違いを作る。高域の折り返しを避ける。
            weights = {0: [1, .22, .10], 4: [1, .35, .08], 24: [1, .30, .18],
                       27: [1, .45, .23], 48: [1, .55, .27], 80: [1, .45, .30],
                       10: [1, .05, .30]}[program]
            wave = np.zeros(len(t))
            for harmonic, weight in enumerate(weights, 1):
                if harmonic * frequency < sr / 2:
                    wave += weight * np.sin(2 * np.pi * frequency * harmonic * t)
            decay = 0.6 if program in (48, 80) else 3.0
            wave *= np.exp(-decay * t)
            amplitude = 0.16
        attack = np.minimum(t / 0.008, 1)
        release = np.minimum((len(t) / sr - t) / 0.025, 1)
        audio[left:right] += amplitude * (note.velocity / 100) * wave * attack * release
    return audio


def arrange(y, sr, config, melody_notes=None):
    """伴奏の拍を指定BPMで配置し、元の音源と同じ長さにミックスする。

    元の音声の速度や音程は変えない。BPM変更は追加する伴奏に作用する。
    """
    y = _audio(y, sr)
    _validate(config)
    duration = len(y) / sr
    rng = np.random.default_rng(config.seed)
    tracks = {"chords": [], "drums": [], "melody": []}
    warnings = []
    beat = 60 / config.bpm
    bar = 4 * beat
    # 7音のスケールは I–V–vi–IV。5音ではスケール上の対応音を使う。
    progression = [0, 4, 5, 3] if len(SCALE_DEFS[config.scale]) == 7 else [0, 3, 4, 2]
    for step, t in enumerate(np.arange(config.beat_offset, duration, beat / 4)):
        step_in_bar = step % 16
        if config.add_chords and step_in_bar in [0, 3, 6, 8, 10, 12, 14]:
            degree = progression[(step // 16) % 4]
            for index, pitch in enumerate(chord_pitches(config.root, config.scale, degree)):
                start = t + index * 0.012
                note = _scaled_note(pitch, start, start + min(.25, beat * .6), 85,
                                    config.chord_gain, duration, rng, config.humanise)
                if note:
                    tracks["chords"].append(note)
        if config.add_drums:
            for pitch, pattern, velocity, length in [(36, [0, 10, 14], 110, .15),
                                                      (38, [4, 12], 100, .15),
                                                      (42, list(range(0, 16, 2)), 75, .05)]:
                if step_in_bar in pattern:
                    note = _scaled_note(pitch, t, t + length, velocity,
                                        config.drum_gain, duration, rng, config.humanise)
                    if note:
                        tracks["drums"].append(note)
    if config.add_melody and config.melody_gain > 0:
        source_notes = extract_melody(y, sr) if melody_notes is None else melody_notes
        for source in source_notes:
            if (not 0 <= source.pitch <= 127 or not math.isfinite(source.start)
                    or not math.isfinite(source.end) or source.start < 0
                    or source.end <= source.start or not 0 <= source.velocity <= 127):
                raise ValueError("メロディの音符データに不正な値があります。")
            note = _scaled_note(source.pitch, source.start, source.end, source.velocity,
                                config.melody_gain, duration, rng, 0)
            if note:
                tracks["melody"].append(note)
        if not tracks["melody"]:
            warnings.append("メロディの音符が見つかりませんでした。単音がはっきりした区間を試してください。")
    midi = pretty_midi.PrettyMIDI(initial_tempo=config.bpm)
    accompaniment = np.zeros(len(y))
    for name, notes in tracks.items():
        program = INSTRUMENTS[config.instrument] if name == "chords" else (0 if name == "drums" else 80)
        instrument = pretty_midi.Instrument(program, is_drum=name == "drums", name=name)
        for note in notes:
            instrument.notes.append(pretty_midi.Note(note.velocity, note.pitch, note.start, note.end))
        if notes:
            midi.instruments.append(instrument)
        accompaniment += _render(notes, sr, len(y), program, name == "drums", config.seed)
    mixed = y * config.original_gain + accompaniment
    peak = np.max(np.abs(mixed))
    if peak > 1:
        mixed /= peak
        accompaniment /= peak
        warnings.append("音割れを防ぐため、ミックス全体の音量を調整しました。")
    return dict(mixed=mixed.astype(np.float32), accompaniment=accompaniment.astype(np.float32),
                midi=midi, notes=tracks, warnings=warnings)


### 2-c. ボタンで操作する画面

音源の取り込み、解析、アレンジ、保存をつなぎます。
音源や設定を変えた際の結果の更新も、このコードで管理しています。

In [ ]:
%%writefile colab_app.py
"""AI作曲アレンジの操作画面。音の処理と画面を分けて読みやすくします。"""
from __future__ import annotations

from dataclasses import asdict
import base64
from html import escape
import json
from pathlib import Path
import tempfile
import zipfile

import ipywidgets as widgets
import numpy as np
import soundfile as sf
from IPython.display import Audio, clear_output, display

from audio_input import AudioInputError, download_youtube, load_audio_file
from music_core import (
    SAMPLE_RATE, NOTE_NAMES, SCALE_DEFS, INSTRUMENTS, ArrangementConfig,
    analyse_audio, arrange, extract_melody, make_demo,
)

SCALE_LABELS = {
    "major": "長調（major）", "natural_minor": "自然短音階（natural minor）",
    "harmonic_minor": "和声的短音階（harmonic minor）", "dorian": "ドリアン",
    "mixolydian": "ミクソリディアン", "pentatonic": "長調の五音音階",
}
MAX_UPLOAD_BYTES = 50 * 1024 * 1024


class MusicArrangerApp:
    """state が現在の音源、controls が画面の入力部品です。"""

    def __init__(self, work_dir=None):
        self.work_dir = Path(work_dir) if work_dir is not None else Path.cwd() / "music_work"
        self.work_dir.mkdir(parents=True, exist_ok=True)
        self.state = {
            "audio": None, "analysis": None, "melody_notes": None,
            "result": None, "artifacts": {}, "source_revision": 0,
            "source": None,
        }
        self.busy = False
        self.controls = self._make_controls()
        self.source_output = widgets.Output()
        self.analysis_output = widgets.Output()
        self.result_output = widgets.Output()
        self.download_output = widgets.Output()
        self.status = widgets.HTML("まずは「デモ音源を用意」を押すと、音源なしで試せます。")
        self.source_status = widgets.HTML("現在の音源：まだ読み込んでいません。")
        self.result_status = widgets.HTML("生成後に、原曲・追加パート・ミックスを聴き比べられます。")
        self._connect_events()
        self.widget = self._make_layout()
        self.widget._music_app = self
        self.widget._music_state = self.state
        self.widget._music_controls = self.controls
        self._update_enabled()

    def _make_controls(self):
        style = {"description_width": "130px"}
        wide = widgets.Layout(width="min(100%, 520px)")
        c = {
            "url": widgets.Text(description="YouTube URL", placeholder="https://www.youtube.com/watch?v=…", style=style, layout=wide),
            "start": widgets.BoundedFloatText(value=0, min=0, max=86400, step=1, description="開始位置（秒）", style=style),
            "duration": widgets.BoundedFloatText(value=20, min=3, max=60, step=1, description="使う長さ（秒）", style=style),
            "youtube": widgets.Button(description="YouTubeから取得", button_style="primary", layout=widgets.Layout(width="200px")),
            "upload": widgets.FileUpload(accept=".wav,.mp3,.m4a,.flac,.ogg,.aac,.webm", multiple=False, description="PCの音源を選択", layout=widgets.Layout(width="200px")),
            "demo": widgets.Button(description="デモ音源を用意", button_style="info", layout=widgets.Layout(width="200px")),
            "analyse": widgets.Button(description="音源を解析", button_style="primary", layout=widgets.Layout(width="200px")),
            "root": widgets.Dropdown(options=NOTE_NAMES, value="C", description="キー（主音）", style=style),
            "scale": widgets.Dropdown(options=[(SCALE_LABELS.get(x, x), x) for x in SCALE_DEFS], value="major", description="スケール", style=style),
            "bpm": widgets.FloatSlider(value=120, min=40, max=220, step=0.1, description="伴奏のBPM", continuous_update=False, style=style, layout=wide),
            "beat_offset": widgets.FloatSlider(value=0, min=0, max=4, step=0.01, description="伴奏の開始（秒）", continuous_update=False, style=style, layout=wide),
            "instrument": widgets.Dropdown(options=list(INSTRUMENTS), value="Electric Piano", description="伴奏の音色", style=style),
            "add_chords": widgets.Checkbox(value=True, description="コード伴奏を追加", indent=False),
            "add_drums": widgets.Checkbox(value=True, description="ドラムを追加", indent=False),
            "add_melody": widgets.Checkbox(value=True, description="抽出メロディを追加", indent=False),
            "original_gain": widgets.FloatSlider(value=0.7, min=0, max=1.5, step=0.05, description="原曲の音量", continuous_update=False, style=style, layout=wide),
            "chord_gain": widgets.FloatSlider(value=0.6, min=0, max=1.5, step=0.05, description="伴奏の音量", continuous_update=False, style=style, layout=wide),
            "drum_gain": widgets.FloatSlider(value=0.8, min=0, max=1.5, step=0.05, description="ドラムの音量", continuous_update=False, style=style, layout=wide),
            "melody_gain": widgets.FloatSlider(value=0.6, min=0, max=1.5, step=0.05, description="メロディの音量", continuous_update=False, style=style, layout=wide),
            "humanise": widgets.FloatSlider(value=0, min=0, max=1, step=0.1, description="ゆらぎの強さ", continuous_update=False, style=style, layout=wide),
            "seed": widgets.BoundedIntText(value=42, min=0, max=2147483647, description="乱数の種", style=style),
            "generate": widgets.Button(description="アレンジを生成", button_style="success", layout=widgets.Layout(width="200px")),
            "download_wav": widgets.Button(description="ミックスWAVを保存", layout=widgets.Layout(width="200px")),
            "download_midi": widgets.Button(description="MIDIを保存", layout=widgets.Layout(width="180px")),
            "download_zip": widgets.Button(description="結果をZIPで保存", button_style="info", layout=widgets.Layout(width="200px")),
        }
        return c

    @staticmethod
    def _panel(title, children):
        return widgets.VBox(
            [widgets.HTML(f"<h3 style='color:#164e63;margin:4px 0 12px'>{title}</h3>"), *children],
            layout=widgets.Layout(border="1px solid #cbd5e1", padding="18px", margin="12px 0", width="100%"),
        )

    def _make_layout(self):
        c = self.controls
        source_tabs = widgets.Tab(children=[
            widgets.VBox([c["url"], c["youtube"], widgets.HTML("動画の指定区間を取得します。取得には時間がかかる場合があります。")]),
            widgets.VBox([c["upload"], widgets.HTML("WAV・MP3など、50 MiB以下の音源を1つ選びます。選ぶと読み込みが始まります。")]),
            widgets.VBox([c["demo"], widgets.HTML("音階と拍が分かる12秒のオリジナル音源です。開始位置・長さの指定は使いません。")]),
        ])
        for i, title in enumerate(["YouTube", "PCの音源", "デモ"]):
            source_tabs.set_title(i, title)
        source_tabs.selected_index = 2
        return widgets.VBox([
            widgets.HTML("<div style='background:#123b4a;color:white;padding:22px;border-radius:12px'><h2 style='margin:0'>AI作曲アレンジ</h2><p style='margin-bottom:0'>音源を読み込み、伴奏とリズムを加えて聴き比べる音楽実験室</p></div>"),
            widgets.HTML("<p>最初は <b>デモ → 音源を解析 → アレンジを生成</b> の順で試してください。<br>自分で制作した音源や、利用許可のある音源を使いましょう。</p>"),
            self._panel("1　音源を用意する", [
                widgets.HBox([c["start"], c["duration"]], layout=widgets.Layout(flex_flow="row wrap")),
                widgets.HTML("YouTube・PC音源は指定した区間だけを使います。入力欄の変更は、次の読み込み時に反映されます。"),
                source_tabs, self.status, self.source_status, self.source_output,
            ]),
            self._panel("2　音源の特徴を調べる", [
                widgets.HTML("テンポ・キー・伴奏の開始位置を推定します。結果は目安です。聴いて違うと感じたら、次の設定で調整してください。"),
                c["analyse"], self.analysis_output,
            ]),
            self._panel("3　アレンジを設定する", [
                widgets.HBox([c["root"], c["scale"]], layout=widgets.Layout(flex_flow="row wrap")),
                c["instrument"], c["bpm"], c["beat_offset"],
                widgets.HTML("BPMと開始位置は<b>追加する伴奏・ドラム</b>の設定です。原曲の速さは変わりません。拍がずれる場合は短い区間で試してください。"),
                widgets.HBox([c["add_chords"], c["add_drums"], c["add_melody"]], layout=widgets.Layout(flex_flow="row wrap")),
                c["original_gain"], c["chord_gain"], c["drum_gain"], c["melody_gain"],
                c["humanise"], c["seed"],
                widgets.HTML("ゆらぎの強さは0で一定、1で最大約18ミリ秒の時刻のずれと音の強弱を加えます。同じ音源・設定・乱数の種で比較できます。"),
                c["generate"], self.result_status,
                widgets.HTML("<small>pYINは1本の音高の流れを推定します。歌声だけを分離する機能ではありません。伴奏は音階とリズムの規則で生成します。内蔵音源の音色は楽器を簡略化したものです。</small>"),
            ]),
            self._panel("4　聴き比べて保存する", [
                self.result_output,
                widgets.HBox([c["download_wav"], c["download_midi"], c["download_zip"]], layout=widgets.Layout(flex_flow="row wrap")),
                widgets.HTML("WAVは聴ける音声、MIDIは追加パートの音符です。MIDIには原曲の録音は入りません。ZIPにはミックス・追加パートのWAV、MIDI、設定をまとめます。ミックスには原曲の音が含まれます。"),
                self.download_output,
            ]),
        ], layout=widgets.Layout(width="100%", max_width="1120px"))

    def _connect_events(self):
        c = self.controls
        c["youtube"].on_click(self._on_youtube)
        c["demo"].on_click(self._on_demo)
        c["upload"].observe(self._on_upload, names="value")
        c["analyse"].on_click(self._on_analyse)
        c["generate"].on_click(self._on_generate)
        for name in asdict(ArrangementConfig()):
            c[name].observe(self._on_config_changed, names="value")
        for name, key in [("download_wav", "mixed.wav"), ("download_midi", "arrangement.mid"), ("download_zip", "arrangement_results.zip")]:
            c[name].on_click(lambda _, filename=key: self._on_download(filename))

    def _update_enabled(self):
        for control in self.controls.values():
            control.disabled = self.busy
        ready = self.state["audio"] is not None
        self.controls["analyse"].disabled = self.busy or not ready
        self.controls["generate"].disabled = self.busy or self.state["analysis"] is None
        for name in ("download_wav", "download_midi", "download_zip"):
            self.controls[name].disabled = self.busy or not self.state["artifacts"]

    def _message(self, text, error=False):
        colour = "#991b1b" if error else "#155e75"
        self.status.value = f"<p role='status' style='color:{colour}'>{escape(text)}</p>"

    def _invalidate_result(self):
        self.state["result"] = None
        self.state["artifacts"] = {}
        self.result_output.clear_output()
        self.download_output.clear_output()
        self.result_status.value = "設定を変更したら「アレンジを生成」を押してください。保存は生成後にできます。"
        self._update_enabled()

    def _on_config_changed(self, change):
        self._invalidate_result()

    def _run(self, action, message):
        if self.busy:
            return
        self.busy = True
        self._update_enabled()
        self._message(message)
        try:
            action()
        except (AudioInputError, ValueError) as exc:
            # audio_inputのエラーは、URLやローカルパスを含まない案内文です。
            note = " 現在の音源はそのままです。" if self.state["audio"] is not None else ""
            self._message(str(exc) + note, error=True)
        except Exception:
            self._message("処理を完了できませんでした。準備セルを実行し直し、短いデモ音源でお試しください。現在の音源は置き換えていません。", error=True)
        finally:
            self.busy = False
            self._update_enabled()

    def _accept_audio(self, audio, kind, start=0):
        audio = np.asarray(audio, dtype=np.float32)
        if audio.ndim != 1 or len(audio) == 0 or not np.all(np.isfinite(audio)):
            raise ValueError("音声を読み込めませんでした。別の音源をお試しください。")
        peak = float(np.max(np.abs(audio)))
        if peak > 1:
            audio = audio / peak
        self._invalidate_result()
        self.state.update(audio=audio, analysis=None, melody_notes=None,
                          source_revision=self.state["source_revision"] + 1,
                          source={"kind": kind, "start_seconds": float(start), "duration_seconds": round(len(audio) / SAMPLE_RATE, 3)})
        self.analysis_output.clear_output()
        self.source_status.value = f"<b>現在の音源：{escape(kind)} / {len(audio) / SAMPLE_RATE:.1f}秒</b>（読み込みごとに解析・生成結果を更新します）"
        with self.source_output:
            clear_output(wait=True)
            display(Audio(audio, rate=SAMPLE_RATE, normalize=False))
        self._message("音源の準備ができました。次に「音源を解析」を押してください。")

    def _on_demo(self, _=None):
        self._run(lambda: self._accept_audio(make_demo(duration=12, sr=SAMPLE_RATE), "デモ"), "デモ音源を作っています…")

    def _on_youtube(self, _=None):
        def prepare():
            self._invalidate_result()
            c = self.controls
            # 一時ファイルを使い、取得に失敗しても現在の音源を保ちます。
            with tempfile.TemporaryDirectory(dir=self.work_dir) as temp_dir:
                path = download_youtube(c["url"].value, Path(temp_dir) / "input.wav", start=c["start"].value, duration=c["duration"].value)
                audio, sr = sf.read(path, dtype="float32")
                if sr != SAMPLE_RATE:
                    raise ValueError("音声の形式を確認できませんでした。音源を再取得してください。")
                self._accept_audio(audio, "YouTube", c["start"].value)
        self._run(prepare, "YouTubeの指定区間を取得しています。しばらくお待ちください…")

    def _on_upload(self, change):
        if not change.get("new"):
            return
        def prepare():
            self._invalidate_result()
            value = change["new"]
            item = next(iter(value.values())) if isinstance(value, dict) else value[0]
            content = bytes(item["content"])
            if not content or len(content) > MAX_UPLOAD_BYTES:
                raise ValueError("音源は50 MiB以下の空でないファイルを選んでください。")
            suffix = Path(item.get("name", "audio.wav")).suffix.lower()
            if suffix not in {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".webm"}:
                raise ValueError("WAV・MP3など、対応する音声ファイルを選んでください。")
            with tempfile.TemporaryDirectory(dir=self.work_dir) as temp_dir:
                path = Path(temp_dir) / ("uploaded" + suffix)
                path.write_bytes(content)
                audio = load_audio_file(path, start=self.controls["start"].value, duration=self.controls["duration"].value, sr=SAMPLE_RATE)
                self._accept_audio(audio, "アップロード", self.controls["start"].value)
        self._run(prepare, "選んだ音源の指定区間を読み込んでいます…")
        # 同じファイルを再選択できるよう、部品内のバイト列も解放します。
        self.controls["upload"].value = ()

    def _on_analyse(self, _=None):
        if self.state["audio"] is None:
            self._message("先に音源を用意してください。", error=True)
            return
        def analyse():
            self._invalidate_result()
            result = analyse_audio(self.state["audio"], SAMPLE_RATE)
            self.state["analysis"] = result
            c = self.controls
            for name, key in [("bpm", "bpm"), ("root", "key"), ("scale", "scale"), ("beat_offset", "beat_offset")]:
                value = result[key]
                if name in {"bpm", "beat_offset"}:
                    value = min(c[name].max, max(c[name].min, float(value)))
                c[name].value = value
            with self.analysis_output:
                clear_output(wait=True)
                display(widgets.HTML(f"<p><b>推定：BPM {c['bpm'].value:.1f} / {escape(str(c['root'].value))} / {escape(SCALE_LABELS.get(c['scale'].value, c['scale'].value))}</b><br>伴奏の開始：{c['beat_offset'].value:.2f}秒<br>{escape(str(result.get('note', '')))}</p>"))
            self._message("解析が終わりました。設定を選び「アレンジを生成」を押してください。")
        self._run(analyse, "テンポ・キー・拍の位置を解析しています。初回は少し時間がかかります…")

    def _config(self):
        return ArrangementConfig(**{name: self.controls[name].value for name in asdict(ArrangementConfig())})

    def _on_generate(self, _=None):
        if self.state["audio"] is None or self.state["analysis"] is None:
            self._message("先に音源を用意して解析してください。", error=True)
            return
        def generate():
            self._invalidate_result()
            config = self._config()
            melody = self.state["melody_notes"]
            if config.add_melody and melody is None:
                self._message("原曲の音高をpYINで推定しています。初回は時間がかかる場合があります…")
                melody = extract_melody(self.state["audio"], SAMPLE_RATE)
                self.state["melody_notes"] = melody
            self._message("伴奏・ドラムを作り、音声と保存ファイルを用意しています…")
            result = arrange(self.state["audio"], SAMPLE_RATE, config, melody_notes=melody if config.add_melody else [])
            artifacts = self._write_artifacts(result, config)
            self.state["result"] = result
            self.state["artifacts"] = artifacts
            self._render_result(result)
            self.result_status.value = "<b>生成できました。</b> 原曲と聴き比べ、気になる設定を1つずつ変えて再生成してみましょう。"
            self._message("生成が終わりました。「4 聴き比べて保存する」で再生・保存できます。")
        self._run(generate, "アレンジの生成を始めます…")

    def _write_artifacts(self, result, config):
        # ファイル名やURLを設定ファイルへ記録しません。
        target = Path(tempfile.mkdtemp(prefix="result_", dir=self.work_dir))
        paths = {name: target / name for name in ("mixed.wav", "accompaniment.wav", "arrangement.mid", "settings.json")}
        sf.write(paths["mixed.wav"], result["mixed"], SAMPLE_RATE, subtype="PCM_16")
        sf.write(paths["accompaniment.wav"], result["accompaniment"], SAMPLE_RATE, subtype="PCM_16")
        result["midi"].write(str(paths["arrangement.mid"]))
        settings = {"version": "1.0.0", "source": self.state["source"], "arrangement": asdict(config), "sample_rate": SAMPLE_RATE,
                    "note": "BPM・開始位置は追加伴奏の設定です。原曲の速度は変えません。"}
        paths["settings.json"].write_text(json.dumps(settings, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        archive = target / "arrangement_results.zip"
        with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
            for name, path in paths.items():
                z.write(path, arcname=name)
            z.writestr("README.txt", "AI作曲アレンジ\n\nmixed.wav：原曲と追加パートを重ねた音声\naccompaniment.wav：追加パートだけの音声\narrangement.mid：追加パートの音符（原曲の録音は含みません）\nsettings.json：今回の設定\n\nミックスには入力音源の音が含まれます。公開・共有の際は音源の利用条件を確認してください。\n")
        paths["arrangement_results.zip"] = archive
        return paths

    def _render_result(self, result):
        with self.result_output:
            clear_output(wait=True)
            for title, audio in [("原曲（使用した区間）", self.state["audio"]), ("追加パートだけ", result["accompaniment"]), ("原曲と追加パートのミックス", result["mixed"])]:
                display(widgets.HTML(f"<h4>{title}</h4>"))
                display(Audio(audio, rate=SAMPLE_RATE, normalize=False))
            for warning in result.get("warnings", []):
                display(widgets.HTML(f"<p style='color:#92400e'>{escape(str(warning))}</p>"))
            self._plot_notes(result)

    @staticmethod
    def _plot_notes(result):
        try:
            import matplotlib.pyplot as plt
        except ImportError:
            return
        fig, ax = plt.subplots(figsize=(10, 2.8))
        colours = {"chords": "#0f766e", "melody": "#2563eb"}
        labels = {"chords": "Chords", "melody": "Extracted melody"}
        for name, colour in colours.items():
            notes = result.get("notes", {}).get(name, [])
            for i, note in enumerate(notes):
                ax.plot([note.start, note.end], [note.pitch, note.pitch], color=colour, linewidth=3, alpha=0.7, label=labels[name] if i == 0 else None)
        ax.set(xlabel="Time (s)", ylabel="MIDI note", title="Added notes (drums omitted)")
        ax.grid(alpha=0.2)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(loc="upper right")
        fig.tight_layout()
        display(widgets.HTML("<p>横軸は時間、縦軸は音の高さです。青は抽出メロディ、緑はコード伴奏を示します。</p>"))
        display(fig)
        plt.close(fig)

    def _on_download(self, filename):
        path = self.state["artifacts"].get(filename)
        if path is None or not Path(path).is_file():
            self._message("設定を確認し、先に「アレンジを生成」を押してください。", error=True)
            return
        with self.download_output:
            clear_output(wait=True)
            try:
                from google.colab import files
            except ImportError:
                # Jupyterの公開フォルダとPythonの作業場所が違っていても
                # 保存できるよう、最大60秒分の生成ファイルをリンクに埋め込みます。
                encoded = base64.b64encode(Path(path).read_bytes()).decode("ascii")
                safe_name = escape(filename, quote=True)
                display(widgets.HTML(
                    f'<a download="{safe_name}" href="data:application/octet-stream;base64,{encoded}">'
                    f'{safe_name} をダウンロード</a>'
                ))
            else:
                files.download(str(path))


def launch_app(work_dir=None):
    """Notebookから呼び出し、表示するVBoxを返します。"""
    return MusicArrangerApp(work_dir=work_dir).widget


## 3. 操作画面を開く

左の ▶ を押すと操作画面が表示されます。まずは画面の①から、デモ音源で試してみましょう。
画面を作り直したい場合も、このセルを再実行できます（前の操作状態はリセットされます）。

In [ ]:
import importlib
import audio_input, music_core, colab_app

# セル2でコードを変更した場合も、その内容を読み直します。
for module in (audio_input, music_core, colab_app):
    importlib.reload(module)

from IPython.display import display
app = colab_app.launch_app()
display(app)


## 4. 設定を一つずつ変えて確かめる

1. デモで「原曲」を0、「メロディ」をオフにし、コードとドラムだけを聴きます。
2. キーをCに固定し、スケールを長調から自然短音階へ変えます。和音の響きはどう変わりますか。
3. ドラムをオフにして、ありの場合と比べます。リズムの印象はどう変わりますか。
4. 「ゆらぎの強さ」を変え、同じ乱数シードと異なる乱数シードで生成して比べます。
5. 単音のはっきりした音源と多重奏の音源で、メロディの音符の取り出しやすさを比べます。

変更した値と、聴いて分かったことをメモしてみましょう。
設定を変えた後は「アレンジを生成」を押して、新しい結果に更新します。

### 保存時の注意

WAVは実際に聴ける音声、MIDIは音の高さ・長さ・楽器などの演奏情報です。
MIDIには原曲の録音は含まれません。ランタイムを切断すると作業ファイルが消えるため、
残したい結果は画面のダウンロードボタンで自分のPCに保存してください。
公開・再配布するNotebookでは「編集 → 出力をすべて消去」を行い、URLやファイル名なども確認しましょう。